# Week 6 — Solved

Interactive visualizations with Plotly: hourly crime distributions, animated temporal patterns, and narrative visualization theory.

## Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, warnings, requests
warnings.filterwarnings('ignore')

# Load pre-merged dataset (from Week 2)
try:
    merged = pd.read_parquet('/home/claude/notebooks/sf_crime_merged.parquet')
    print(f'Loaded merged dataset: {len(merged):,} rows')
except FileNotFoundError:
    print('Run Week 2 first to generate the merged parquet, or re-fetch here.')
    # Fallback: re-fetch just 'now' data for standalone use
    import requests
    API_NOW = 'https://data.sfgov.org/resource/wg3w-h783.json'
    def fetch_all(url, params=None, chunk=50_000):
        if params is None: params = {}
        records, offset = [], 0
        while True:
            r = requests.get(url, params={**params, '$limit': chunk, '$offset': offset}, timeout=120)
            r.raise_for_status(); batch = r.json()
            if not batch: break
            records.extend(batch); offset += chunk
            if len(batch) < chunk: break
        return pd.DataFrame(records)
    raw = fetch_all(API_NOW)
    raw['date'] = pd.to_datetime(raw.get('incident_date', pd.NaT), errors='coerce')
    raw['year'] = raw['date'].dt.year
    raw['month'] = raw['date'].dt.month
    raw['weekday'] = raw['date'].dt.day_name()
    raw['hour'] = pd.to_datetime(raw.get('incident_time',''), format='%H:%M', errors='coerce').dt.hour
    raw['category'] = raw.get('incident_category', '')
    raw['district'] = raw.get('police_district', '')
    raw['lat'] = pd.to_numeric(raw.get('latitude', None), errors='coerce')
    raw['lon'] = pd.to_numeric(raw.get('longitude', None), errors='coerce')
    merged = raw.dropna(subset=['date']).copy()

FOCUS_CRIMES = [
    'Assault', 'Burglary', 'Drug Offense', 'Larceny Theft',
    'Motor Vehicle Theft', 'Robbery', 'Vandalism', 'Weapons Offense',
    'Arson', 'Disorderly Conduct', 'Fraud', 'Prostitution',
    'Sex Offense', 'Stolen Property', 'Suspicious Occ', 'Traffic Violation Arrest'
]
FOCUS_CRIMES = [c for c in FOCUS_CRIMES if c in merged['category'].unique()]
df_focus = merged[merged['category'].isin(FOCUS_CRIMES)].copy()
print(f'Focus crimes available: {len(FOCUS_CRIMES)}')


In [ ]:
import plotly.express as px
import plotly.graph_objects as go


## Exercise 1.1 — Explanatory data visualization

**Three key elements of explanatory visualization:**
1. **Overview first** — give the viewer the big picture before details
2. **Zoom and filter** — let users narrow down to what interests them
3. **Details on demand** — reveal specifics only when asked (hover, click)

An example following these principles: [Our World in Data's COVID vaccination tracker](https://ourworldindata.org/covid-vaccinations). Overview = world map with colors. Zoom = select specific countries. Details = hover for exact numbers and dates.

**Explanatory vs Exploratory:** Exploratory visualization is for the analyst to understand data themselves — it can be messy and provisional. Explanatory visualization is designed to communicate a specific insight to an audience — it must be clear, accurate, and guide the reader toward a conclusion.

## Exercise 2.1 — Interactive hourly crime distributions

In [ ]:
# Normalize hourly distributions for each focus crime
df_focus = df_focus.dropna(subset=['hour'])
df_focus['hour'] = df_focus['hour'].astype(int)

hour_counts = df_focus.groupby(['category','hour']).size().reset_index(name='count')
totals = hour_counts.groupby('category')['count'].transform('sum')
hour_counts['normalized'] = hour_counts['count'] / totals

fig = px.bar(
    hour_counts, x='hour', y='normalized', color='category',
    barmode='overlay', opacity=0.6,
    title='Normalized Hourly Crime Distributions — All Focus Crimes',
    labels={'normalized': 'P(crime at this hour)', 'hour': 'Hour of Day', 'category': 'Crime Type'},
    range_x=[-0.5, 23.5], range_y=[0, 0.15]
)

# Start all traces hidden so user can toggle one by one
fig.update_traces(visible='legendonly')

fig.update_layout(
    legend=dict(orientation='v', x=1.01, y=1),
    xaxis=dict(tickmode='linear', dtick=2),
    height=500
)

fig.write_html('interactive_hourly.html')
fig.show()
print('Saved interactive_hourly.html')


**Observations:**
- Prostitution and Drug Offense peak strongly at night (11pm–3am).
- Fraud and Traffic Violation Arrest peak during business hours (9am–5pm).
- Larceny Theft has a broad daytime peak, consistent with opportunistic theft in busy areas.
- Interactivity makes it easy to compare two specific crimes — much harder to do with static multi-line plots.

## Exercise 2.2 — Animated line chart by year

In [ ]:
# Compute normalized hourly distribution per year per crime
year_hour = df_focus.groupby(['year','category','hour']).size().reset_index(name='count')
year_totals = year_hour.groupby(['year','category'])['count'].transform('sum')
year_hour['normalized'] = year_hour['count'] / year_totals

fig = px.line(
    year_hour,
    x='hour', y='normalized', color='category',
    animation_frame='year',
    title='Hourly Crime Distributions by Year (animated)',
    labels={'normalized': 'P(crime at this hour)', 'hour': 'Hour', 'category': 'Crime Type'},
    range_y=[0, 0.20]
)

# Start all except one hidden
fig.update_traces(visible='legendonly')

fig.update_layout(
    legend=dict(orientation='v', x=1.01, y=1),
    height=500
)

fig.write_html('animated_hourly.html')
fig.show()
print('Saved animated_hourly.html')


**Observations on the animation:**
- The 2020 frame shows notably different patterns for many crime types — the COVID lockdowns altered when people were out.
- Drug Offense becomes noisier in recent years as total counts decrease, making the distribution more variable.
- Larceny Theft patterns are remarkably stable year-over-year, suggesting a consistent structural driver (retail and commute patterns).

**Animated lines vs animated bars:** Lines make temporal evolution (year-to-year change) easier to see because the eye can track the line's shape changing. Bars are better for comparing absolute magnitudes at a given moment.

## Exercise 3.1–3.2 — Narrative visualization questions

**Oxford English Dictionary definition of narrative:** A narrative is an account of a series of events, facts, etc., given in order and with the establishment of connections between them.

**Favorite visualization from Section 3:** The *New York Times Obama Budget Proposal* interactive — it uses a treemap structure with animation to show proportional changes. It achieves overview (the whole budget as a visual space), zoom (click into departments), and details on demand (hover for exact numbers).

**Connecting to Week 6 visualizations:**

Our bar chart and animated line chart sit toward the *reader-driven* end of the Segel & Heer spectrum — users can toggle any crime type, explore any year, zoom freely. They're closest to a 'drill-down story' or 'free-form exploration'.

To push them toward author-driven, we would: (1) pre-select which two crimes are shown, (2) add annotation arrows pointing to the COVID dip with text explaining it, (3) disable the legend toggle so the reader sees what we choose.

**For the Board of Supervisors:** A 'Magazine' or 'Annotated Chart' genre — author-driven, single scrolling page, with key findings highlighted. They need clear takeaways, not exploratory freedom.

**For a peer-reviewed journal:** A static figure with small multiples — one panel per crime type, all visible simultaneously, with a detailed figure caption. Interactivity is impossible in a PDF.